<a href="https://colab.research.google.com/github/wohecha/HuggingFace-Unit1/blob/main/notebooks/unit8/unit8_part2-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
# Install ViZDoom deps from
# https://github.com/mwydmuch/ViZDoom/blob/master/doc/Building.md#-linux

sudo apt install build-essential zlib1g-dev libsdl2-dev libjpeg-dev \
nasm tar libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev \
libopenal-dev timidity libwildmidi-dev unzip ffmpeg

# Boost libraries
sudo apt install libboost-all-dev

# Lua binding dependencies
sudo apt install liblua5.1-dev

## Download and install Miniconda
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

In [2]:
%pwd

'/content'

In [3]:
%ls -lah


total 150M
drwxr-xr-x 1 root root 4.0K Feb 22 18:42 ./
drwxr-xr-x 1 root root 4.0K Feb 22 18:40 ../
drwxr-xr-x 4 root root 4.0K Feb  6 14:31 .config/
-rw-r--r-- 1 root root 150M Dec 16 21:40 Miniconda3-latest-Linux-x86_64.sh
drwxr-xr-x 1 root root 4.0K Feb  6 14:31 sample_data/


In [ ]:
%%bash
# It appears there's an issue with the Conda environment and pip paths,
# leading to "ModuleNotFoundError". A full runtime restart is required first.
# After the runtime restarts, please run this cell.

# --- Clean up previous Miniconda installation for a fresh start ---
rm -rf /usr/local/bin/conda /usr/local/condabin /usr/local/etc/profile.d/conda.sh /usr/local/envs /usr/local/lib/python* /usr/local/share/conda

# The previous Miniconda download resulted in 'Miniconda3-latest-Linux-x86_64.sh'.
# Ensure we use the correct file for installation.
chmod +x /content/Miniconda3-latest-Linux-x86_64.sh
sudo /content/Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local -u # -u for unattended installation

# --- Diagnostic commands: Check if conda files exist after installation ---
echo "Contents of /usr/local:"
ls -la /usr/local
echo "\nContents of /usr/local/bin:"
ls -la /usr/local/bin
echo "\nContents of /usr/local/etc/profile.d:"
ls -la /usr/local/etc/profile.d

# Initialize conda for the current shell session and ensure paths are set.
# Use the full path to _conda and explicitly source ~/.bashrc.
/usr/local/_conda init bash
source ~/.bashrc

# Accept Conda terms of service (needed for conda create/install)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Create the environment, specifying Python 3.12 and the classic solver
conda create --name rl_viz python=3.12 --yes --solver classic

# Install required packages within the rl_viz environment using 'conda run'
# 'conda run' executes a command in a specified environment without activating it in the current shell.
conda run -n rl_viz pip install faster-fifo==1.5.2
conda run -n rl_viz pip install vizdoom
conda run -n rl_viz pip install sample-factory==2.1.1
conda run -n rl_viz pip install sample_factory

# Now execute the Python script using the rl_viz environment's python.
conda run -n rl_viz python -c "
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)

def register_vizdoom_models():
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)

def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

register_vizdoom_components()
env = \"doom_health_gathering_supreme\"
cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=8\",
        \"--num_envs_per_worker=4\",
        \"--train_for_env_steps=40000\"
        ]
    )
status = run_rl(cfg)
"

In [7]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 18:48 .
drwxr-xr-x 3 root root 4.0K Feb 22 18:48 ..
drwxr-xr-x 2 root root 4.0K Feb 22 18:48 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 18:48 config.json
-rw-r--r-- 1 root root  16K Feb 22 18:48 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 18:48 .summary


In [6]:
!cat /content/train_dir/default_experiment/sf_log.txt

[2026-02-22 18:48:11,044][04392] Saving configuration to /content/train_dir/default_experiment/config.json...
[2026-02-22 18:48:11,044][04392] Rollout worker 0 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 1 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 2 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 3 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 4 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 5 uses device cpu
[2026-02-22 18:48:11,045][04392] Rollout worker 6 uses device cpu
[2026-02-22 18:48:11,046][04392] Rollout worker 7 uses device cpu
[2026-02-22 18:48:11,131][04392] Using GPUs [0] for process 0 (actually maps to GPUs [0])
[2026-02-22 18:48:11,131][04392] InferenceWorker_p0-w0: min num requests: 2
[2026-02-22 18:48:11,158][04392] Starting all processes...
[2026-02-22 18:48:11,158][04392] Starting process learner_proc0
[2026-02-22 18:48:11,196][04392] Starting all processes...
[2026-02-22 18

In [8]:
%%bash
conda run -n rl_viz python -c "
from sample_factory.enjoy import enjoy
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

env = \"doom_health_gathering_supreme\"
cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=1\",
        \"--save_video\",
        \"--no_render\",
        \"--max_num_episodes=10\"
        ],
    evaluation=True
    )
"

In [ ]:
print(cfg)

In [ ]:
status = enjoy(cfg)

In [9]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 18:48 .
drwxr-xr-x 3 root root 4.0K Feb 22 18:48 ..
drwxr-xr-x 2 root root 4.0K Feb 22 18:48 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 18:48 config.json
-rw-r--r-- 1 root root  16K Feb 22 18:48 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 18:48 .summary


In [ ]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 16:54 .
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 ..
drwxr-xr-x 2 root root 4.0K Feb 22 16:54 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 16:54 config.json
-rw-r--r-- 1 root root  16K Feb 22 16:54 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 .summary


In [ ]:
from base64 import b64encode
from IPython.display import HTML

mp4 = open('/content/train_dir/default_experiment/replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

FileNotFoundError: [Errno 2] No such file or directory: '/content/train_dir/default_experiment/replay.mp4'

In [ ]:
!pip install huggingface_hub

In [ ]:
!hf auth login

In [ ]:
%%bash

conda run -n rl_viz python -c "
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.enjoy import enjoy

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)

def register_vizdoom_models():
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)

def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

# Register the ViZDoom components before creating the environment
register_vizdoom_components()

env = \"doom_health_gathering_supreme\"
hf_username = \"seb-835\" # insert your HuggingFace username here

cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=1\",
        \"--save_video\",
        \"--no_render\",
        \"--max_num_episodes=10\",
        \"--max_num_frames=100000\",
        \"--push_to_hub\",
        f\"--hf_repository={hf_username}/rl_course_vizdoom_health_gathering_supreme\"
    ],
    evaluation=True
)
status = enjoy(cfg)
"